# ReCAHS — Reproducible Five-Seed Baseline Training

Bu notebook, B4 PatchTST baseline modelini `SEEDS = [7, 42, 1234, 2026, 3407]` ile yeniden üretilebilir biçimde eğitir. Daha önce tamamlanmış seed sonuçları özet CSV üzerinden algılanır ve tekrar çalıştırılmaz.

Önemli düzeltme: Time-Series-Library commit `4e938a...` içindeki `--seed` argümanı ana eğitim seed'ini değiştirmiyor; `run.py` değeri sabit `2021` yapıyor. Bu notebook `run.py` dosyasını idempotent biçimde `RECAHS_SEED` ortam değişkenini kullanacak şekilde düzeltir ve uygulanan diff'i kaydeder.


In [ ]:
# 1) AYARLAR
from pathlib import Path

REPO_URL = 'https://github.com/didemneda/regime-aware-head-pruning.git'
BASE_BRANCH = 'didem/patchtst-baseline'
WORK_BRANCH = 'feat/reproducible-pipeline'
REPO_DIR = Path('/content/regime-aware-head-pruning')
PROJECT_DIR = Path('/content/drive/MyDrive/BIL401_Regime_Head_Pruning')
TSLIB_DIR = Path('/content/Time-Series-Library')
TSLIB_COMMIT = '4e938a1767106324dd753b2a44832bf870a0252e'

# Nihai beş-seed deney listesi.
SEEDS = [7, 42, 1234, 2026, 3407]
RUN_TRAINING = True

DATA_URL = 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv'
print('Seeds:', SEEDS)
print('Time-Series-Library commit:', TSLIB_COMMIT)

In [ ]:
# 2) DRIVE, REPO VE SABİT TSLIB COMMIT'İNİ HAZIRLA
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import sys

def run(command, cwd=None, env=None, check=True):
    print('$', ' '.join(map(str, command)))
    result = subprocess.run(
        list(map(str, command)), cwd=str(cwd) if cwd else None,
        env=env, text=True, capture_output=True
    )
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Komut başarısız (kod={result.returncode})')
    return result

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--branch', BASE_BRANCH, REPO_URL, REPO_DIR])
local_branch = run(
    ['git', 'show-ref', '--verify', '--quiet', f'refs/heads/{WORK_BRANCH}'],
    cwd=REPO_DIR, check=False
)
if local_branch.returncode == 0:
    run(['git', 'switch', WORK_BRANCH], cwd=REPO_DIR)
else:
    run(['git', 'switch', '-c', WORK_BRANCH], cwd=REPO_DIR)

if not (TSLIB_DIR / '.git').exists():
    run(['git', 'clone', 'https://github.com/thuml/Time-Series-Library.git', TSLIB_DIR])
run(['git', 'fetch', 'origin'], cwd=TSLIB_DIR)
run(['git', 'checkout', '--force', TSLIB_COMMIT], cwd=TSLIB_DIR)
actual_commit = run(['git', 'rev-parse', 'HEAD'], cwd=TSLIB_DIR).stdout.strip()
assert actual_commit == TSLIB_COMMIT
print('Sabit TSLib commit doğrulandı:', actual_commit)

In [ ]:
# 3) GEREKLİ PAKETLERİ KUR
# PatchTST için kullanılan önceki notebook kurulumuyla uyumlu tutuldu.
!pip install -q patool sktime scikit-base --no-deps
!pip install -q einops --no-deps
!pip install -q reformer-pytorch --no-deps
!pip install -q local-attention --no-deps
!pip install -q hyper-connections --no-deps
!pip install -q axial-positional-embedding --no-deps
!pip install -q product-key-memory --no-deps
!pip install -q colt5-attention --no-deps

print('Paket kurulumu tamamlandı.')

In [ ]:
# 4) ETTh1 VERİSİNİ İNDİR, DOĞRULA VE DRIVE'A KAYDET
import hashlib
import shutil
import urllib.request

import pandas as pd

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for block in iter(lambda: file.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

drive_data_dir = PROJECT_DIR / 'data'
drive_data_dir.mkdir(parents=True, exist_ok=True)
drive_data_path = drive_data_dir / 'ETTh1.csv'

if not drive_data_path.exists():
    temp_path = Path('/content/ETTh1.download.csv')
    urllib.request.urlretrieve(DATA_URL, temp_path)
    downloaded = pd.read_csv(temp_path)
    expected_columns = ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
    if downloaded.columns.tolist() != expected_columns:
        raise ValueError(f'Beklenmeyen ETTh1 sütunları: {downloaded.columns.tolist()}')
    if len(downloaded) != 17420:
        raise ValueError(f'Beklenmeyen ETTh1 satır sayısı: {len(downloaded)}')
    shutil.copy2(temp_path, drive_data_path)

data = pd.read_csv(drive_data_path)
assert data.columns.tolist() == ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
assert len(data) == 17420
dataset_sha256 = sha256_file(drive_data_path)

tslib_data_dir = TSLIB_DIR / 'dataset' / 'ETDataset' / 'ETT-small'
tslib_data_dir.mkdir(parents=True, exist_ok=True)
tslib_data_path = tslib_data_dir / 'ETTh1.csv'
shutil.copy2(drive_data_path, tslib_data_path)

print('ETTh1 shape:', data.shape)
print('ETTh1 SHA256:', dataset_sha256)
print('TSLib veri yolu:', tslib_data_path)

In [ ]:
# 5) TSLIB SEED HATASINI KONTROLLÜ VE İDEMPOTENT BİÇİMDE DÜZELT
run_py = TSLIB_DIR / 'run.py'
source = run_py.read_text(encoding='utf-8')

old_seed = '    fix_seed = 2021\n'
new_seed = "    fix_seed = int(os.environ.get('RECAHS_SEED', '2021'))\n"
if old_seed in source:
    source = source.replace(old_seed, new_seed, 1)
elif new_seed not in source:
    raise RuntimeError('run.py içindeki seed satırı beklenen biçimde değil.')

old_torch = '    torch.manual_seed(fix_seed)\n'
new_torch = (
    '    torch.manual_seed(fix_seed)\n'
    '    if torch.cuda.is_available():\n'
    '        torch.cuda.manual_seed_all(fix_seed)\n'
    '    torch.backends.cudnn.deterministic = True\n'
    '    torch.backends.cudnn.benchmark = False\n'
)
if 'torch.cuda.manual_seed_all(fix_seed)' not in source:
    if old_torch not in source:
        raise RuntimeError('run.py içindeki torch seed satırı bulunamadı.')
    source = source.replace(old_torch, new_torch, 1)

run_py.write_text(source, encoding='utf-8')
diff_text = run(['git', 'diff', '--', 'run.py'], cwd=TSLIB_DIR).stdout
patch_dir = REPO_DIR / 'patches'
patch_dir.mkdir(parents=True, exist_ok=True)
patch_path = patch_dir / 'time_series_library_seed.patch'
patch_path.write_text(diff_text, encoding='utf-8')

assert "RECAHS_SEED" in run_py.read_text(encoding='utf-8')
assert diff_text.strip(), 'Seed patch diff oluşmadı.'
print(diff_text)
print('Patch kaydedildi:', patch_path)

In [ ]:
# 6) SEED BAZLI B4 EĞİTİM FONKSİYONU
import json
import os
from datetime import datetime, timezone

import numpy as np

MULTISEED_DIR = PROJECT_DIR / 'multiseed' / 'ETTh1'
MULTISEED_DIR.mkdir(parents=True, exist_ok=True)

def train_seed(seed):
    seed_dir = MULTISEED_DIR / f'seed_{seed}'
    checkpoint_root = seed_dir / 'checkpoints'
    seed_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    model_id = f'ETTh1_336_96_dm128_h8_seed{seed}'

    command = [
        sys.executable, '-u', 'run.py',
        '--task_name', 'long_term_forecast',
        '--is_training', '1',
        '--root_path', './dataset/ETDataset/ETT-small/',
        '--data_path', 'ETTh1.csv',
        '--model_id', model_id,
        '--model', 'PatchTST',
        '--data', 'ETTh1',
        '--features', 'M',
        '--seq_len', '336', '--label_len', '48', '--pred_len', '96',
        '--enc_in', '7', '--dec_in', '7', '--c_out', '7',
        '--e_layers', '3', '--d_layers', '1', '--factor', '3',
        '--d_model', '128', '--d_ff', '256', '--n_heads', '8',
        '--batch_size', '32', '--train_epochs', '10', '--patience', '3',
        '--learning_rate', '0.0001',
        '--num_workers', '0',
        '--checkpoints', str(checkpoint_root),
        '--des', f'baseline_b4_seed{seed}', '--itr', '1',
    ]

    env = os.environ.copy()
    env['RECAHS_SEED'] = str(seed)
    env['PYTHONHASHSEED'] = str(seed)
    env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

    log_path = seed_dir / 'training.log'
    print('$', ' '.join(map(str, command)))
    started = datetime.now(timezone.utc)
    with log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command, cwd=str(TSLIB_DIR), env=env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
        )
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
        returncode = process.wait()
    finished = datetime.now(timezone.utc)

    if returncode != 0:
        raise RuntimeError(f'seed={seed} eğitimi başarısız. Log: {log_path}')

    metric_candidates = sorted(
        (TSLIB_DIR / 'results').glob(f'*{model_id}*/metrics.npy'),
        key=lambda p: p.stat().st_mtime, reverse=True
    )
    checkpoint_candidates = sorted(checkpoint_root.glob('**/checkpoint.pth'))
    if len(metric_candidates) != 1:
        raise RuntimeError(f'seed={seed}: metrics.npy sayısı {len(metric_candidates)}')
    if len(checkpoint_candidates) != 1:
        raise RuntimeError(f'seed={seed}: checkpoint sayısı {len(checkpoint_candidates)}')

    metrics = np.load(metric_candidates[0])
    record = {
        'dataset': 'ETTh1', 'seed': seed, 'method': 'unpruned_baseline',
        'test_mae': float(metrics[0]), 'test_mse': float(metrics[1]),
        'test_rmse': float(metrics[2]), 'test_mape': float(metrics[3]),
        'test_mspe': float(metrics[4]),
        'dataset_sha256': dataset_sha256,
        'tslib_commit': TSLIB_COMMIT,
        'repo_commit': run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR).stdout.strip(),
        'checkpoint': str(checkpoint_candidates[0]),
        'checkpoint_sha256': sha256_file(checkpoint_candidates[0]),
        'started_at_utc': started.isoformat(),
        'finished_at_utc': finished.isoformat(),
        'duration_seconds': (finished - started).total_seconds(),
    }
    (seed_dir / 'run_metadata.json').write_text(
        json.dumps(record, indent=2), encoding='utf-8'
    )
    return record

print('Eğitim fonksiyonu hazır.')

In [ ]:
# 7) BEŞ SEED'İ ÇALIŞTIR VE ÖNCEKİ SONUÇLARLA BİRLEŞTİR

summary_path = MULTISEED_DIR / "baseline_multiseed_summary.csv"

if summary_path.exists():
    previous_summary = pd.read_csv(summary_path)
    completed_seeds = set(previous_summary["seed"].astype(int))
    print("Daha önce tamamlanan seed'ler:", sorted(completed_seeds))
else:
    previous_summary = pd.DataFrame()
    completed_seeds = set()

records = []

if RUN_TRAINING:
    for seed in SEEDS:
        if seed in completed_seeds:
            print(f"seed={seed} zaten tamamlanmış, atlanıyor.")
            continue

        print(f"\n===== SEED {seed} BAŞLIYOR =====")
        record = train_seed(seed)
        records.append(record)
else:
    print("RUN_TRAINING=False; eğitim başlatılmadı.")

new_summary = pd.DataFrame(records)

if previous_summary.empty:
    combined_summary = new_summary
elif new_summary.empty:
    combined_summary = previous_summary
else:
    combined_summary = pd.concat(
        [previous_summary, new_summary],
        ignore_index=True,
    )

if not combined_summary.empty:
    combined_summary = (
        combined_summary
        .drop_duplicates(
            subset=["dataset", "seed", "method"],
            keep="last",
        )
        .sort_values(["dataset", "seed", "method"])
        .reset_index(drop=True)
    )

    combined_summary.to_csv(summary_path, index=False)

    display(
        combined_summary[
            [
                "dataset",
                "seed",
                "test_mse",
                "test_mae",
                "duration_seconds",
            ]
        ]
    )

    baseline_rows = combined_summary[
        combined_summary["method"] == "unpruned_baseline"
    ]

    print("\nÇoklu-seed baseline özeti:")
    print(
        "Test MSE:",
        f"{baseline_rows['test_mse'].mean():.6f}",
        "±",
        f"{baseline_rows['test_mse'].std(ddof=1):.6f}",
    )
    print(
        "Test MAE:",
        f"{baseline_rows['test_mae'].mean():.6f}",
        "±",
        f"{baseline_rows['test_mae'].std(ddof=1):.6f}",
    )
    print("\nÖzet CSV:", summary_path)

print("\nRepo değişiklikleri:")
run(["git", "status", "--short"], cwd=REPO_DIR)

